# Hypoxia signature model — single-dataset discovery + scorer\n\nBuilt incrementally per `CLAUDE.md`. Scope for this session: seed list → discovery (Eqs 1, 2, 5 + Monte Carlo) → scorer (Eq 7 + HS). Adapters, Cox validation, and sigQC are out of scope here."

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## Seed genes (Buffa 2010, HNSCC training network, set A)

The ten seed genes exactly as printed in the paper, hardcoded — not a file to
load. One of the ten (`AK3L1`) was later renamed by HGNC to `AK4`; modern
datasets use the new symbol, so we keep both forms and resolve at lookup time
against whatever gene index the actual dataset has.

In [ ]:
# Ten seed genes, literal names as printed in Buffa et al. 2010.
SEED_GENES = [
    "ADM", "AK3L1", "BNIP3", "CA9", "ENO1",
    "HK2", "LDHA", "PGK1", "SLC2A1", "VEGFA",
]

# AK3L1 -> AK4 was a formal HGNC rename, not a casual alias. Exact-string
# matching against a modern dataset (Ensembl/GEO/TCGA-annotated) will miss
# AK3L1 entirely and silently drop that seed unless we also try AK4.
SEED_ALIASES = {
    "AK3L1": "AK4",
}


def resolve_seed_genes(seed_genes: list[str], available_genes) -> dict[str, str]:
    """Map each seed (paper name) to whichever symbol is present in `available_genes`.

    Tries the literal paper name first, then its known alias. Seeds matching
    neither are left out of the returned dict -- callers should check for
    missing seeds rather than assume all ten resolve.
    """
    available = set(available_genes)
    resolved = {}
    for seed in seed_genes:
        if seed in available:
            resolved[seed] = seed
        elif seed in SEED_ALIASES and SEED_ALIASES[seed] in available:
            resolved[seed] = SEED_ALIASES[seed]
    return resolved

## Discovery step 1: affinity gate + membership (Eqs 1, 2)

For one seed gene, Spearman-correlate it against every gene in the dataset.
Eq 1 is a hard pass/fail gate: a gene passes only if its correlation with the
seed exceeds `theta_t`, the critical correlation for significance at `alpha`
(Bonferroni-corrected across all `m` genes tested against this seed). Eq 2
turns that gate into a membership weight: 0 if gated out, otherwise scaling
with `|rho|`.

**Flagging a real ambiguity** (per CLAUDE.md: these equations are
reconstructed from descriptive text, not the paper's actual image-rendered
formulas): Eq 1 as written compares `rho^2 > theta_t`, with `theta_t`
described as "the correlation needed for significance." It's not stated
whether `theta_t` already means the *squared* critical correlation or the
*unsquared* one compared against a squared statistic. This doesn't actually
change which genes pass, though: `rho^2 > r_crit^2` and `|rho| > r_crit` are
the same test. So below I just compare `|rho| > theta_t` directly, with
`theta_t` being the critical correlation magnitude itself — simpler, and
identical in outcome to the paper's version under either reading.

Eq 2's exact functional form isn't given either, just the qualitative
behavior ("increases with `|rho|`"). The simplest form consistent with that
is `gamma = delta * |rho|`, which is what's implemented.

In [ ]:
def critical_correlation(n: int, m: int, alpha: float = 0.05) -> float:
    """Critical |correlation| for significance at `alpha`, Bonferroni-corrected
    across `m` comparisons, for a sample of size `n` (theta_t in Eq 1).

    Converts the corrected alpha into a critical t-statistic (two-tailed,
    df = n - 2), then maps that back to a correlation value via the standard
    r <-> t relationship: t = r * sqrt((n - 2) / (1 - r^2)).
    """
    alpha_corrected = alpha / m
    df = n - 2
    t_crit = stats.t.ppf(1 - alpha_corrected / 2, df)
    return t_crit / np.sqrt(df + t_crit ** 2)


def affinity_and_membership(seed_expr: pd.Series, matrix: pd.DataFrame,
                             theta_t: float) -> tuple[pd.Series, pd.Series]:
    """Eq 1 (affinity) + Eq 2 (membership) for one seed against every gene in `matrix`.

    seed_expr : expression vector for the seed gene (samples on the index)
    matrix    : samples x genes matrix, correlated column-by-column against seed_expr
    theta_t   : critical |correlation| from `critical_correlation`

    Returns (delta, gamma), each indexed by gene:
      delta : 1.0 where |rho| > theta_t, else 0.0   (Eq 1 affinity gate)
      gamma : delta * |rho|                          (Eq 2 membership)
    """
    rho = matrix.corrwith(seed_expr, method="spearman")
    delta = (rho.abs() > theta_t).astype(float)
    gamma = delta * rho.abs()
    return delta, gamma